# Transformer via DSL — `arch` declaration (v3)

One `compile()` call produces an `ArchDef`. One `arch.interpreter('Transformer')`
returns an `ArchInterpreter` with both `.run_algebra()` and `.run_coalgebra()`.

What the DSL declares:

- **Semiring** — the contract op that all morphisms share.
- **Morphisms** — typed sorts, einsum equations, per-morphism op overrides.
  Morphisms are the arrows between sorts — parametric maps in the sense of
  Gavranović et al. (2024). Arity declarations (`unary`, `binary`, etc.)
  control how many data arguments a morphism receives.
- **Paths** — compositions of morphisms. `residual` combinator adds a skip
  connection around the whole path.
- **Fan-out** — `kv` runs `k_proj` and `v_proj` in parallel.
- **`[fan_name]`** — augment combinator: runs the fan, merges its dict output
  into y (the weight dict), leaves x unchanged. This eliminates bundle assembly.
- **`arch Transformer:`** — algebra cases (tree fold) with `morphisms =`
  declarations that derive cells automatically. ALL algebra cases are now derived
  — no Python cells needed for the algebra side.

What remains as Python:

- Utility functions (softmax, layer_norm, gelu, causal_mask).
- Per-morphism op implementations — arbitrary numpy, referenced by dotted names.
- `stream_cell` — coalgebra KV cache management (only remaining Python cell).
- Weight initialization.

In [1]:
import sys, os
import numpy as np

ROOT = os.path.abspath('..')
sys.path.insert(0, ROOT)
sys.modules.pop("engine", None)

from engine import compile, CoalgResult

np.set_printoptions(precision=4, suppress=True)

## 1. Hyperparameters and utilities

These are pure Python — they don't belong in the DSL. The DSL's job is to
declare the *structure* of the architecture; these are the *atoms* it composes.

In [2]:
V = 128
D = 48
N = 4
H = D // N
F = D * 3
L = 2
S = 8
EPS = 1e-5
SCALE = H ** -0.5

assert D % N == 0

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def softmax_masked(x: np.ndarray, mask: np.ndarray | None = None, axis: int = -1) -> np.ndarray:
    if mask is not None:
        x = x + mask
    return softmax(x, axis=axis)

def layer_norm(x: np.ndarray, gamma: np.ndarray, beta: np.ndarray, eps: float = EPS) -> np.ndarray:
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return ((x - mu) / np.sqrt(var + eps)) * gamma + beta

def gelu(x: np.ndarray) -> np.ndarray:
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

def causal_mask(t: int) -> np.ndarray:
    return np.triu(np.full((t, t), -1e9), 1)

print({'vocab': V, 'd_model': D, 'heads': N, 'd_head': H, 'd_ff': F, 'layers': L, 'seq_len': S})

{'vocab': 128, 'd_model': 48, 'heads': 4, 'd_head': 12, 'd_ff': 144, 'layers': 2, 'seq_len': 8}


## 2. Ops, cell functions, and DSL architecture

Everything the DSL references lives in a single namespace:

- **Morphism ops** — `(compiled_eq, x, y, temp) -> result`. Arity determines
  which arguments are active:
  - `binary` (default): `op(eq, x, y, temp=)` where x is data, y is weights
  - `unary`: `op(eq, x, temp=)` — ignores y (e.g. gelu, layer_norm)
  - `pointwise`: `op(eq, x, y, temp=)` — y is data, not weights (e.g. residual add)

- **Augment combinator** — `[kv]` in a path runs the `kv` fan on x and merges
  its dict output into y. After `[kv]`, y contains `k_proj` and `v_proj` keys
  in addition to the original weight dict fields. This replaces `_build_attn_bundle`.

- **Cell functions** — only needed for cases the DSL can't derive. With `[kv]`
  augment, ALL algebra cases can now use `morphisms =` or `cell=identity` — no
  Python algebra cells remain.

The `arch Transformer:` block declares both algebra and coalgebra sides:
- `input` uses `cell=identity` — base case, returns payload[0]
- `attn_residual` uses `morphisms = attn` — `ln1` pre-norm + `[kv]` augment handle bundle assembly
- `ffn_residual` uses `morphisms = ffn` — `ln2` pre-norm + `up act down` + residual, fully DSL-derived
- `final_norm` uses `morphisms = ln` — no Python cell needed

In [3]:
# ---------------------------------------------------------------------------
# Per-morphism op implementations (plain numpy)
# ---------------------------------------------------------------------------

def q_proj_op(eq, x, w, temp=0.0):
    return np.einsum(eq, x, w['Wq']) + w['bq']

def k_proj_op(eq, x, w, temp=0.0):
    return np.einsum(eq, x, w['Wk']) + w['bk']

def v_proj_op(eq, x, w, temp=0.0):
    return np.einsum(eq, x, w['Wv']) + w['bv']

def out_proj_op(eq, x, w, temp=0.0):
    return np.einsum(eq, x, w['Wo']) + w['bo']

def score_op(eq, q, y, temp=0.0):
    """After [kv] augment, y contains k_proj and v_proj merged into weight dict."""
    return np.einsum(eq, q, y['k_proj']) * y.get('scale', SCALE)

def normalize_op(eq, scores, y, temp=0.0):
    return softmax_masked(scores, y.get('mask'))

def mix_op(eq, probs, y, temp=0.0):
    """After [kv] augment, y contains v_proj merged into weight dict."""
    return np.einsum(eq, probs, y['v_proj'])

def up_op(eq, x, w, temp=0.0):
    return np.einsum(eq, x, w['W1']) + w['b1']

def down_op(eq, x, w, temp=0.0):
    return np.einsum(eq, x, w['W2']) + w['b2']

# --- unary ops (arity unary — ignore y) ---

def act_op(eq, x, temp=0.0):
    return gelu(x)

def ln_op(eq, x, temp=0.0):
    """Layer norm with identity scale/shift — used for final_norm."""
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + EPS)

def ln1_op(eq, x, w, temp=0.0):
    """Binary layer norm: reads gamma/beta from weight dict under ln1_g / ln1_b.
    Used as the pre-norm step in the attn path."""
    return layer_norm(x, w['ln1_g'], w['ln1_b'])

def ln2_op(eq, x, w, temp=0.0):
    """Binary layer norm: reads gamma/beta from weight dict under ln2_g / ln2_b.
    Used as the pre-norm step in the ffn path."""
    return layer_norm(x, w['ln2_g'], w['ln2_b'])

def identity_op(eq, x, y=None, temp=0.0):
    return x


# ---------------------------------------------------------------------------
# Coalgebra cell — streaming with KV cache (only remaining Python cell)
# ---------------------------------------------------------------------------

def stream_cell(state, event, params, temp):
    paths = params['paths']
    mode, token = event
    pos = state['pos']
    caches = state['caches']

    x = params['tok_embed'][token] + params['pos_enc'][pos]
    x = x[None, :]

    next_caches = []
    for layer_idx, w in enumerate(params['layer_weights']):
        x_norm = layer_norm(x, w['ln1_g'], w['ln1_b'])
        bundle = {
            'Wq': w['Wq'], 'bq': w['bq'],
            'Wk': w['Wk'], 'bk': w['bk'],
            'Wv': w['Wv'], 'bv': w['bv'],
            'Wo': w['Wo'], 'bo': w['bo'],
            'scale': SCALE, 'mask': None,
        }
        kv = paths['kv'](x_norm, bundle, 0.0)
        K_all = np.concatenate([caches[layer_idx]['K'], kv['k_proj']], axis=0)
        V_all = np.concatenate([caches[layer_idx]['V'], kv['v_proj']], axis=0)
        # score_op and mix_op read from k_proj / v_proj keys (same names as fan outputs)
        bundle['k_proj'] = K_all
        bundle['v_proj'] = V_all
        x = x + paths['read'](x_norm, bundle, 0.0)
        x = x + paths['mlp'](layer_norm(x, w['ln2_g'], w['ln2_b']), w, 0.0)
        next_caches.append({'K': K_all, 'V': V_all})

    logits = (layer_norm(x, params['final_ln_g'], params['final_ln_b']) @ params['unembed'])[0]
    next_state = {'pos': pos + 1, 'caches': next_caches}

    if mode == 'cache':
        return CoalgResult('cache_only', [pos], [next_state])
    return CoalgResult('emit', [pos], [next_state], output=logits)


# ---------------------------------------------------------------------------
# DSL architecture
# ---------------------------------------------------------------------------

DSL_SOURCE = """
semiring attn:
    contract = ops.identity

sort model, q, scores, probs, mixed, kv, ff

# --- binary morphisms (default): x is data, y is weight dict ---
morphism ln1       : model  -> model   via "sd->sd"        op ops.ln1
morphism ln2       : model  -> model   via "sd->sd"        op ops.ln2
morphism q_proj    : model  -> q       via "sd,ndh->snh"   op ops.q_proj
morphism score     : q      -> scores  via "snh,tnh->nst"  op ops.score
morphism normalize : scores -> probs   via "nst->nst"       op ops.normalize
morphism mix       : probs  -> mixed   via "nst,tnh->snh"  op ops.mix
morphism out_proj  : mixed  -> model   via "snh,nhd->sd"   op ops.out_proj
morphism k_proj    : model  -> kv      via "sd,ndh->snh"   op ops.k_proj
morphism v_proj    : model  -> kv      via "sd,ndh->snh"   op ops.v_proj

morphism up   : model -> ff     via "sd,df->sf"   op ops.up
morphism down : ff    -> model  via "sf,fd->sd"   op ops.down

# --- unary morphisms: ignore y, operate on x only ---
morphism act : ff    -> ff     via "sf->sf"      op ops.act   arity unary
morphism ln  : model -> model  via "sd->sd"      op ops.ln    arity unary

# --- paths with combinators ---
# read: inner attention computation (used for tracing and by stream_cell)
path read = q_proj score normalize mix out_proj
# attn: full attention path — ln1 pre-norm, [kv] augment merges k/v into y, residual wrap
path attn = ln1 [kv] q_proj score normalize mix out_proj  residual
# ffn: full FFN path — ln2 pre-norm, up act down, residual wrap
path ffn  = ln2 up act down  residual
path mlp  = up act down

fan kv = k_proj & v_proj

arch Transformer:
    algebra:
        case input:         recursive=0  data=1  cell=identity
        case attn_residual: recursive=1  data=1  morphisms = attn
        case ffn_residual:  recursive=1  data=1  morphisms = ffn
        case final_norm:    recursive=1  data=1  morphisms = ln
    coalgebra:
        cell = ops.stream_cell
        case cache_only: recursive=1  data=1  output=0
        case emit:       recursive=1  data=1  output=1
"""

arch = compile(DSL_SOURCE, {
    'ops': type('ns', (), {
        'identity':     staticmethod(identity_op),
        'q_proj':       staticmethod(q_proj_op),
        'k_proj':       staticmethod(k_proj_op),
        'v_proj':       staticmethod(v_proj_op),
        'out_proj':     staticmethod(out_proj_op),
        'score':        staticmethod(score_op),
        'normalize':    staticmethod(normalize_op),
        'mix':          staticmethod(mix_op),
        'up':           staticmethod(up_op),
        'act':          staticmethod(act_op),
        'down':         staticmethod(down_op),
        'ln':           staticmethod(ln_op),
        'ln1':          staticmethod(ln1_op),
        'ln2':          staticmethod(ln2_op),
        'stream_cell':  staticmethod(stream_cell),
    })(),
})

print("Compiled architecture:")
print(f"  paths: {sorted(arch.paths.keys())}")
print(f"  arch:  Transformer (algebra + coalgebra)")
print(f"  derived cells: input (identity), attn_residual (attn), ffn_residual (ffn), final_norm (ln)")
print(f"  explicit cells: stream_cell only")

Compiled architecture:
  paths: ['act', 'attn', 'down', 'ffn', 'k_proj', 'kv', 'ln', 'ln1', 'ln2', 'mix', 'mlp', 'normalize', 'out_proj', 'q_proj', 'read', 'score', 'up', 'v_proj']
  arch:  Transformer (algebra + coalgebra)
  derived cells: input (identity), attn_residual (attn), ffn_residual (ffn), final_norm (ln)
  explicit cells: stream_cell only


## 3. Path inspection

`ArchDef.explain()` shows the canonical morphism sequence for any named path.

In [4]:
print(arch.explain("attn"))
print()
print(arch.explain("ffn"))
print()
print(arch.explain("read"))

Path: attn
Normal form: ln1 q_proj score normalize mix out_proj
  1. ln1  [sd->sd]
  2. q_proj  [sd,ndh->snh]
  3. score  [snh,tnh->nst]
  4. normalize  [nst->nst]
  5. mix  [nst,tnh->snh]
  6. out_proj  [snh,nhd->sd]

Path: ffn
Normal form: ln2 up act down
  1. ln2  [sd->sd]
  2. up  [sd,df->sf]
  3. act  [sf->sf]
  4. down  [sf,fd->sd]

Path: read
Normal form: q_proj score normalize mix out_proj
  1. q_proj  [sd,ndh->snh]
  2. score  [snh,tnh->nst]
  3. normalize  [nst->nst]
  4. mix  [nst,tnh->snh]
  5. out_proj  [snh,nhd->sd]


## 4. Parameters

In [5]:
rng = np.random.default_rng(7)

def normal(shape, std=0.02):
    return rng.normal(0.0, std, shape)

def init_block(d: int, n: int, h: int, f: int) -> dict:
    return {
        'Wq': normal((n, d, h)),
        'bq': np.zeros((n, h)),
        'Wk': normal((n, d, h)),
        'bk': np.zeros((n, h)),
        'Wv': normal((n, d, h)),
        'bv': np.zeros((n, h)),
        'Wo': normal((n, h, d)),
        'bo': np.zeros(d),
        'ln1_g': np.ones(d),
        'ln1_b': np.zeros(d),
        'ln2_g': np.ones(d),
        'ln2_b': np.zeros(d),
        'W1': normal((d, f)),
        'b1': np.zeros(f),
        'W2': normal((f, d)),
        'b2': np.zeros(d),
        'scale': SCALE,
    }

layer_weights = [init_block(D, N, H, F) for _ in range(L)]
tok_embed = normal((V, D))
pos_enc = normal((64, D))
final_ln_g = np.ones(D)
final_ln_b = np.zeros(D)

# explicit tied output projection
unembed = tok_embed.T

## 5. Algebra forward pass

`arch.interpreter('Transformer')` returns an `ArchInterpreter`.
`.run_algebra()` uses the algebra functor. All four algebra cases have their
cells derived automatically — `input` via `cell=identity`, the other three
via `morphisms =` declarations. No Python cell functions needed.

In [6]:
def build_program(blocks, x0, mask=None):
    """Build the algebra tree: input -> (attn + ffn) per layer -> final norm.

    mask is added to each weight dict at call time so the attn path can read
    it from y. scale is already in the weight dict from init_block.
    """
    node = ('input', [x0], [])
    for w in blocks:
        w_aug = {**w, 'mask': mask}
        node = ('ffn_residual', [w_aug], [
            ('attn_residual', [w_aug], [node])
        ])
    node = ('final_norm', [None], [node])
    return node

def transformer_forward(token_ids):
    t = len(token_ids)
    x0 = tok_embed[token_ids] + pos_enc[:t]
    interp = arch.interpreter('Transformer')
    x = interp.run_algebra(build_program(layer_weights, x0, causal_mask(t)), lambda n: n)
    return x @ unembed

token_ids = rng.integers(0, V, size=S)
logits = transformer_forward(token_ids)
print("token_ids:", token_ids)
print("logits shape:", logits.shape)

token_ids: [  0  27  72  83  50  41 104  94]
logits shape: (8, 128)


## 6. Path trace

`ArchDef.trace()` executes a path step-by-step, showing the morphism name,
einsum equation, and output shape at each step.

In [7]:
x0 = tok_embed[token_ids] + pos_enc[:S]
w0 = layer_weights[0]

# Build a trace bundle for the read sub-path:
# ln1 pre-norm + [kv] augment are steps in the attn path but not in read.
# We replicate them here so the trace has k_proj and v_proj available in y.
x_norm = ln1_op("sd->sd", x0, w0)
kv_result = arch.paths['kv'](x_norm, w0, 0.0)
trace_bundle = {**w0, **kv_result, 'mask': causal_mask(S)}

print("Attention path trace (read sub-path, after ln1 + kv augment):")
for name, eq, shape, _ in arch.trace("read", x_norm, trace_bundle):
    print(f"  {name:<12} eq={str(eq):<16} shape={shape}")

print("\nFFN path trace:")
for name, eq, shape, _ in arch.trace("mlp", x0, w0):
    print(f"  {name:<12} eq={str(eq):<16} shape={shape}")

Attention path trace (read sub-path, after ln1 + kv augment):
  input        eq=None             shape=(8, 48)
  q_proj       eq=sd,ndh->snh      shape=(8, 4, 12)
  score        eq=snh,tnh->nst     shape=(4, 8, 8)
  normalize    eq=nst->nst         shape=(4, 8, 8)
  mix          eq=nst,tnh->snh     shape=(8, 4, 12)
  out_proj     eq=snh,nhd->sd      shape=(8, 48)

FFN path trace:
  input        eq=None             shape=(8, 48)
  up           eq=sd,df->sf        shape=(8, 144)
  act          eq=sf->sf           shape=(8, 144)
  down         eq=sf,fd->sd        shape=(8, 48)


## 7. Coalgebra streaming

Same `ArchInterpreter`, now calling `.run_coalgebra()`. The coalgebra cell
is bound at the functor level (`cell = ops.stream_cell`) — this is the most
complex remaining Python cell, handling per-layer KV cache concatenation.

In [8]:
prompt = token_ids[:6].tolist()
events = [('cache', tok) for tok in prompt[:-1]] + [('emit', prompt[-1])]

interp = arch.interpreter('Transformer', params={
    'tok_embed': tok_embed,
    'pos_enc': pos_enc,
    'layer_weights': layer_weights,
    'final_ln_g': final_ln_g,
    'final_ln_b': final_ln_b,
    'unembed': unembed,
})

stream_outputs, final_state = interp.run_coalgebra(
    state={
        'pos': 0,
        'caches': [{'K': np.zeros((0, N, H)), 'V': np.zeros((0, N, H))} for _ in range(L)],
    },
    token_iter=events,
)

full_last = transformer_forward(np.array(prompt))[-1]
stream_last = stream_outputs[-1]

print("prompt:", prompt)
print("emitted outputs:", len(stream_outputs))
print("final position:", final_state['pos'])
print("max abs diff vs batch forward:", np.max(np.abs(stream_last - full_last)))

prompt: [0, 27, 72, 83, 50, 41]
emitted outputs: 1
final position: 6
max abs diff vs batch forward: 1.1102230246251565e-16


## 8. Summary

This notebook defines a multi-head, multi-layer transformer through a single DSL block:

```
arch Transformer:
    algebra:
        case input:         recursive=0  data=1  cell=identity
        case attn_residual: recursive=1  data=1  morphisms = attn
        case ffn_residual:  recursive=1  data=1  morphisms = ffn
        case final_norm:    recursive=1  data=1  morphisms = ln
    coalgebra:
        cell = ops.stream_cell
        case cache_only: recursive=1  data=1  output=0
        case emit:       recursive=1  data=1  output=1
```

**What the refactored DSL now handles structurally:**
- `morphism ln1 / ln2` — binary layer norms that read gamma/beta from the weight dict
- `path attn = ln1 [kv] q_proj score normalize mix out_proj  residual` — the `[kv]`
  augment combinator runs the `kv` fan on x and merges `k_proj`/`v_proj` into y,
  replacing the `_build_attn_bundle` function and `attn_cell` entirely
- `path ffn = ln2 up act down  residual` — `ln2` pre-norm before FFN, residual wrap
- `cell=identity` on `input` — built-in, returns payload[0]

**All algebra cases are now DSL-derived — zero Python algebra cells remain.**

**What still requires Python cells (1 of 5 cases):**
- `stream_cell` — KV cache concatenation across layers (coalgebra only)

**Remaining gaps that would eliminate stream_cell:**
- Accumulate morphisms (`accumulate cat`): would handle KV cache in the coalgebra runner
- Requires the sort system to evolve from labels to structured type descriptors

The algebra/coalgebra agreement holds: max abs diff between batch forward and
streaming last-token logits is at machine epsilon.